In [2]:
import sys
sys.path.append('../src')
from data_prep import prepare_full_dataset

df = prepare_full_dataset("C:\\Users\\Acer\\Documents\\GitHub\\energy-demand-forecast\\data\\demand_2025_hourly.csv")
df.head()

,value,datetime_utc,tz_time,geo_id,geo_name,hour,dayofweek,month,dayofyear,is_weekend,hour_sin,hour_cos,month_sin,month_cos,lag_24h,lag_168h,is_blackout
datetime,,,,,,,,,,,,,,,,,
2025-01-08 00:00:00+01:00,315876.0,2025-01-07T23:00:00Z,2025-01-07T23:00:00.000Z,8741,Península,0,2,1,8,0,0.000000,1.000000,0.5,0.866025,278464.0,275088.0,0
2025-01-08 01:00:00+01:00,290801.0,2025-01-08T00:00:00Z,2025-01-08T00:00:00.000Z,8741,Península,1,2,1,8,0,0.258819,0.965926,0.5,0.866025,257973.0,266231.0,0
2025-01-08 02:00:00+01:00,275867.0,2025-01-08T01:00:00Z,2025-01-08T01:00:00.000Z,8741,Península,2,2,1,8,0,0.500000,0.866025,0.5,0.866025,245482.0,251434.0,0
2025-01-08 03:00:00+01:00,269501.0,2025-01-08T02:00:00Z,2025-01-08T02:00:00.000Z,8741,Península,3,2,1,8,0,0.707107,0.707107,0.5,0.866025,241077.0,237297.0,0
2025-01-08 04:00:00+01:00,269468.0,2025-01-08T03:00:00Z,2025-01-08T03:00:00.000Z,8741,Península,4,2,1,8,0,0.866025,0.500000,0.5,0.866025,242624.0,227683.0,0


In [3]:
import pandas as pd

# Últim mes (desembre 2025) com a test, la resta com a train
train = df[df.index < "2025-12-01"]
test = df[df.index >= "2025-12-01"]

print(f"Train: {train.shape[0]} files ({train.index.min()} a {train.index.max()})")
print(f"Test: {test.shape[0]} files ({test.index.min()} a {test.index.max()})")

Train: 7848 files (2025-01-08 00:00:00+01:00 a 2025-11-30 23:00:00+01:00)
Test: 744 files (2025-12-01 00:00:00+01:00 a 2025-12-31 23:00:00+01:00)


In [4]:
from xgboost import XGBRegressor

features = ['hour', 'dayofweek', 'month', 'is_weekend', 
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
            'lag_24h', 'lag_168h', 'is_blackout']

X_train, y_train = train[features], train['value']
X_test, y_test = test[features], test['value']

model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [5]:
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

mape = mean_absolute_percentage_error(y_test, preds) * 100
rmse = root_mean_squared_error(y_test, preds)

print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.0f} MW")

MAPE: 4.44%
RMSE: 22706 MW


In [9]:
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
import pandas as pd

features = ['hour', 'dayofweek', 'month', 'is_weekend', 
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
            'lag_24h', 'lag_168h', 'is_blackout']

X = df[features]
y = df['value']

tscv = TimeSeriesSplit(n_splits=5)
results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    print(f"Fold {fold+1}:")
    print(f"  Train: {X_train.index.min()} → {X_train.index.max()} ({len(X_train)} files)")
    print(f"  Test:  {X_test.index.min()} → {X_test.index.max()} ({len(X_test)} files)")
    
    
    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    # Excloure el dia de l'apagada del càlcul de mètriques
    mask = (X_test['is_blackout'] == 0).values
    
    if X_test['is_blackout'].sum() > 0:
        print(f"  ⚠️ Fold {fold+1} inclou el dia de l'apagada ({X_test['is_blackout'].sum()} files excloses del càlcul)")
    
    mape = mean_absolute_percentage_error(y_test[mask], preds[mask]) * 100
    rmse = root_mean_squared_error(y_test[mask], preds[mask])
    
    results.append({'fold': fold+1, 'MAPE': mape, 'RMSE': rmse})
    print(f"Fold {fold+1}: MAPE={mape:.2f}%, RMSE={rmse:.0f}")

results_df = pd.DataFrame(results)
print(f"\nMAPE mitjà: {results_df['MAPE'].mean():.2f}%")
print(f"RMSE mitjà: {results_df['RMSE'].mean():.0f}")

Fold 1:
  Train: 2025-01-08 00:00:00+01:00 → 2025-03-08 15:00:00+01:00 (1432 files)
  Test:  2025-03-08 16:00:00+01:00 → 2025-05-07 08:00:00+02:00 (1432 files)
  ⚠️ Fold 1 inclou el dia de l'apagada (24 files excloses del càlcul)
Fold 1: MAPE=9.44%, RMSE=37495
Fold 2:
  Train: 2025-01-08 00:00:00+01:00 → 2025-05-07 08:00:00+02:00 (2864 files)
  Test:  2025-05-07 09:00:00+02:00 → 2025-07-06 00:00:00+02:00 (1432 files)
Fold 2: MAPE=5.95%, RMSE=25275
Fold 3:
  Train: 2025-01-08 00:00:00+01:00 → 2025-07-06 00:00:00+02:00 (4296 files)
  Test:  2025-07-06 01:00:00+02:00 → 2025-09-03 16:00:00+02:00 (1432 files)
Fold 3: MAPE=3.92%, RMSE=17175
Fold 4:
  Train: 2025-01-08 00:00:00+01:00 → 2025-09-03 16:00:00+02:00 (5728 files)
  Test:  2025-09-03 17:00:00+02:00 → 2025-11-02 07:00:00+01:00 (1432 files)
Fold 4: MAPE=3.03%, RMSE=12279
Fold 5:
  Train: 2025-01-08 00:00:00+01:00 → 2025-11-02 07:00:00+01:00 (7160 files)
  Test:  2025-11-02 08:00:00+01:00 → 2025-12-31 23:00:00+01:00 (1432 files)
Fold 5